In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertModel, DataCollatorWithPadding, AutoTokenizer
import pandas as pd
import json
%load_ext autoreload
%autoreload 2

from ml_article_tagging.config import PROCESSED_DATA_DIR


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [5]:
# Loading our pretrained LLM
llm = BertModel.from_pretrained("allenai/scibert_scivocab_uncased")
tokenizer = AutoTokenizer.from_pretrained("allenai/scibert_scivocab_uncased")

/home/dhiran/miniconda3/envs/madewithml/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at allenai/scibert_scivocab_uncased were not used when initializing BertModel: ['cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.predictions.decoder.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.decoder.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expecte

In [6]:
llm.config # type: ignore

BertConfig {
  "_name_or_path": "allenai/scibert_scivocab_uncased",
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.28.1",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 31090
}

In [7]:
llm.named_parameters

<bound method Module.named_parameters of BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(31090, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (d

In [8]:
class ArticleDataset(Dataset):
    def __init__(self, data):
        self.encodings = data["encodings"]
        self.labels = data["labels"]
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):

        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(self.labels[idx])

        return item

In [ ]:
def create_dataloader(dataset, tokenizer,  batch_size, shuffle=True):
    dataset = ArticleDataset(dataset)

    collator = DataCollatorWithPadding(tokenizer=tokenizer)
    
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        collate_fn=collator
    )
    
    return dataloader

In [ ]:
class SciBERTClassifier(nn.Module):
    def __init__(self, llm, dropput_p, num_classes):
        super().__init__()
        
        self.llm = llm
        self.dropout = nn.Dropout(dropput_p)
        self.classifier = nn.Linear(llm.config.hidden_size, num_classes)
    
    def forward(self, x):
        input_ids= x["input_ids"]
        attention_mask = x["attention_mask"]

        outputs = self.llm(
            input_ids=input_ids, 
            attention_mask=attention_mask
            )
        CLS = outputs.last_hidden_state[:, 0] # preferred in modern implementation

        z = self.dropout(CLS)
        z = self.classifier(z)

        return z

In [ ]:
with open(PROCESSED_DATA_DIR / "class_to_index.json", "r") as f:
    class_to_index = json.load(f)

In [ ]:
# model initialization
model = SciBERTClassifier(
    llm,
    dropput_p=0.2,
    num_classes=len(class_to_index)
)

In [ ]:
model.named_parameters

<bound method Module.named_parameters of SciBERTClassifier(
  (llm): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(31090, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): 

In [ ]:
train_data = torch.load(PROCESSED_DATA_DIR/ "train_data.pt")

print(train_data.keys())
print(train_data["encodings"].keys())

dict_keys(['encodings', 'labels'])
dict_keys(['input_ids', 'token_type_ids', 'attention_mask'])


In [ ]:
train_dataloader = create_dataloader(
    dataset=train_data,
    tokenizer=tokenizer,
    batch_size=32
)

batch = next(iter(train_dataloader))

for k, v in batch.items():
    print(k, v.shape)

You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


input_ids torch.Size([32, 55])
token_type_ids torch.Size([32, 55])
attention_mask torch.Size([32, 55])
labels torch.Size([32])


In [ ]:
logits = model(batch)
logits.shape

torch.Size([32, 4])